In [36]:

import pandas as pd
import os

# Universal file loader function
def load_file(file_path):
    ext = os.path.splitext(file_path)[1].lower()

    if ext == '.csv':
        df = pd.read_csv(file_path)
    elif ext in ['.xls', '.xlsx']:
        df = pd.read_excel(file_path, engine='openpyxl')
    elif ext == '.json':
        df = pd.read_json(file_path)
    elif ext == '.parquet':
        df = pd.read_parquet(file_path)
    else:
        raise ValueError(f"Unsupported file type: {ext}")

    print(f" Loaded file: {file_path}")
    print(f"Shape: {df.shape}")
    print("\nDynamic Attribute Summary:")
    for col in df.columns:
        dtype = df[col].dtype
        sample_values = df[col].dropna().head(3).tolist()
        print(f"Column: {col} | Type: {dtype} | Sample: {sample_values}")

    return df

# Example usage
file_path = r"unified_claims_model_dataset.csv"  # Replace with your file path
df = load_file(file_path)  # df now holds your dataset


 Loaded file: unified_claims_model_dataset.csv
Shape: (7500, 18)

Dynamic Attribute Summary:
Column: patient_id | Type: object | Sample: ['PT100000', 'PT100001', 'PT100002']
Column: age_group | Type: object | Sample: ['0-17', '35-49', '18-34']
Column: sex | Type: object | Sample: ['M', 'M', 'F']
Column: region | Type: object | Sample: ['South', 'North', 'North']
Column: provider_id | Type: float64 | Sample: [9639472848.0, 6303021087.0, 9670335754.0]
Column: speciality | Type: object | Sample: ['Podiatry', 'Plastic Surgery', 'Infectious Disease']
Column: npi | Type: float64 | Sample: [9639472848.0, 9670335754.0, 3381395514.0]
Column: procedure_code | Type: object | Sample: ['11721', '15734', '96372']
Column: procedure_descrption | Type: object | Sample: ['Debridement nail', 'Scar revision', 'Therapeutic injection']
Column: category | Type: object | Sample: ['Procedure', 'Procedure', 'Procedure']
Column: date_of_service | Type: object | Sample: ['2023-10-13', '2024-03-23', '2023-11-23']


In [11]:
import hashlib
# Helper function for hashing IDs
def hash_id(x):
    return hashlib.sha256(str(x).encode()).hexdigest()


In [52]:

# Create dim_patient from original dataset
dim_patient = df[['patient_id', 'age_group', 'sex', 'region']].drop_duplicates().reset_index(drop=True)

# Add surrogate key
dim_patient['patient_sk'] = range(1, len(dim_patient) + 1)

# Hash patient_id for privacy
def hash_id(x):
    return hash(str(x))

dim_patient['patient_id_hashed'] = dim_patient['patient_id'].apply(hash_id)
# Define mapping for age groups to descriptive categories
age_mapping = {
    '0-17': 'Child',
    '18-34': 'Young Adult',
    '35-49': 'Adult',
    '50-64': 'Middle Age',
    '65+': 'Senior'
}

# Replace age_group values with descriptive categories in the same column
df['age_group'] = df['age_group'].map(age_mapping)


# Select final columns
dim_patient = dim_patient[['patient_sk', 'patient_id_hashed', 'age_group', 'sex', 'region']]
dim_patient.to_csv('dim_patient.csv', index=False)
# Preview
print(dim_patient.head())


   patient_sk    patient_id_hashed age_group sex   region
0           1  6121963012507210171      0-17   M    South
1           2  8047848451011608872     35-49   M    North
2           3 -6547671582812561179     18-34   F    North
3           4  1434062762517281172      0-17   M     East
4           5 -1749962996992915902       65+   M  Central


In [60]:

dim_provider = df[['provider_id', 'speciality', 'npi']].drop_duplicates().reset_index(drop=True)
dim_provider['provider_sk'] = range(1, len(dim_provider) + 1)
dim_provider['provider_id'] = dim_provider['provider_id'].fillna(-1)
dim_provider['npi'] = dim_provider['npi'].fillna(-1)
dim_provider = dim_provider[['provider_sk', 'provider_id', 'speciality', 'npi']]

default_provider_sk = 0
if -1 in dim_provider['provider_id'].values:
    dim_provider = pd.concat([
        pd.DataFrame([[default_provider_sk, -1, 'Unknown', -1]], columns=['provider_sk', 'provider_id', 'speciality', 'npi']),
        dim_provider
    ], ignore_index=True)

dim_provider.to_csv('dim_provider.csv', index=False)


dim_provider.head()

,provider_sk,provider_id,speciality,npi
0,0,-1.000000e+00,Unknown,-1.000000e+00
1,1,9.639473e+09,Podiatry,9.639473e+09
2,2,6.303021e+09,Plastic Surgery,-1.000000e+00
3,3,9.670336e+09,Infectious Disease,9.670336e+09
4,4,3.381396e+09,ENT,3.381396e+09


In [54]:
print(cpt_df.columns)
print(icd_df.columns)


Index(['com.medigy.persist.reference.type.clincial.CPT.code', 'label'], dtype='object')
Index(['Diagnosis', 'ICD-10 Code', 'ICD Description', 'Similarity Score',
       'Justification', 'Alternative Suggestions', 'Needs Review'],
      dtype='object')


In [59]:
import pandas as pd

# Load datasets
claims_df = pd.read_csv("unified_claims_model_dataset.csv")
cpt_df = pd.read_csv('cpt4.csv')
icd_df = pd.read_csv('icd10_mapped_output.csv')

# Rename CPT and ICD columns for easier merge
cpt_df = cpt_df.rename(columns={
    'com.medigy.persist.reference.type.clincial.CPT.code': 'procedure_code',
    'label': 'cpt_descrption'
})

icd_df = icd_df.rename(columns={
    'ICD-10 Code': 'procedure_code',
    'ICD Description': 'icd_descrption'
})

# Keep all rows from main dataset
dim_procedure = claims_df[['procedure_code', 'procedure_descrption', 'category']].copy()

# Merge CPT and ICD descriptions
dim_procedure = dim_procedure.merge(
    cpt_df[['procedure_code', 'cpt_descrption']],
    on='procedure_code',
    how='left'
)
dim_procedure = dim_procedure.merge(
    icd_df[['procedure_code', 'icd_descrption']],
    on='procedure_code',
    how='left'
)

# Fill missing descriptions: main > CPT > ICD > default
dim_procedure['procedure_descrption'] = (
    dim_procedure['procedure_descrption']
    .combine_first(dim_procedure['cpt_descrption'])
    .combine_first(dim_procedure['icd_descrption'])
    .fillna('Unknown Procedure')
)

# Drop helper columns
dim_procedure = dim_procedure[['procedure_code', 'procedure_descrption', 'category']]

# Clean procedure_code
dim_procedure['procedure_code'] = dim_procedure['procedure_code'].astype(str).str.strip()
dim_procedure['procedure_code'].replace({'': 'UNKNOWN', 'nan': 'UNKNOWN'}, inplace=True)

# Generate row-level surrogate key (unique per row)
dim_procedure['proc_sk'] = range(1, len(dim_procedure) + 1)

# Save to CSV
dim_procedure.to_csv('dim_procedure.csv', index=False)

print(f"dim_procedure table created successfully! Total records: {len(dim_procedure)}")
print(dim_procedure.head())


dim_procedure table created successfully! Total records: 7500
  procedure_code      procedure_descrption    category  proc_sk
0          11721          Debridement nail   Procedure        1
1          15734             Scar revision   Procedure        2
2          96372     Therapeutic injection   Procedure        3
3          92511         Speech audiometry  Diagnostic        4
4          87070  Culture, bacteria, other         Lab        5


In [19]:

# # Map patient_sk from dim_patient
# patient_map = dict(zip(df['patient_id'], dim_patient['patient_sk']))
# fact_claim['patient_sk'] = fact_claim['patient_id'].map(patient_map)

# # Map provider_sk from dim_provider
# provider_map = dict(zip(dim_provider['provider_id'], dim_provider['provider_sk']))
# fact_claim['provider_sk'] = fact_claim['provider_id'].map(provider_map)

# # Map proc_sk from dim_procedure
# proc_map = dict(zip(dim_procedure['procedure_code'], dim_procedure['proc_sk']))
# fact_claim['proc_sk'] = fact_claim['procedure_code'].map(proc_map)

# # Add claim_sk
# fact_claim['claim_sk'] = range(1, len(fact_claim) + 1)
df.head()

,patient_id,age_group,sex,region,provider_id,speciality,npi,procedure_code,procedure_descrption,category,date_of_service,billed_amount,paid_amount,adjudication_status,Provider Organization Name (Legal Business Name),Provider Last Name (Legal Name),Provider First Name,is_fake_npi
0,PT100000,0-17,M,South,9.639473e+09,Podiatry,9.639473e+09,11721,Debridement nail,Procedure,2023-10-13,5944.48,5880.39,Paid,NaN,ROSENBLOOM,FRANK,0
1,PT100001,35-49,M,North,6.303021e+09,Plastic Surgery,NaN,15734,Scar revision,Procedure,2024-03-23,2014.75,1926.35,Paid,NaN,MINOR,SUMMER,0
2,PT100002,18-34,F,North,9.670336e+09,Infectious Disease,9.670336e+09,96372,Therapeutic injection,Procedure,2023-11-23,NaN,5168.72,Paid,NaN,RICHERT,EMMA,0
3,PT100003,0-17,M,East,3.381396e+09,ENT,3.381396e+09,92511,Speech audiometry,Diagnostic,2024-12-05,502.79,NaN,Partially Paid,FULL CIRCLE RECOVERY OF CIRCLEVILLE,NaN,NaN,0
4,PT100004,65+,M,Central,7.892771e+09,Infectious Disease,7.892771e+09,87070,NaN,Lab,2024-08-23,418.11,NaN,Paid,NaN,TREVISAN,MARIE,0


In [20]:

import pandas as pd

# Assuming df is your main claims dataset
# Load updated dataset with age categories
df = pd.read_csv("unified_claims_model_dataset.csv")

# --- Create fact_claim ---
fact_claim = df[['patient_id', 'provider_id', 'procedure_code', 'date_of_service',
                 'billed_amount', 'paid_amount', 'adjudication_status']].copy()

# Add surrogate key for fact table
fact_claim['claim_sk'] = range(1, len(fact_claim) + 1)

# # Map patient_sk from dim_patient
# patient_map = dict(zip(dim_patient['patient_id_hashed'], dim_patient['patient_sk']))
# fact_claim['patient_sk'] = fact_claim['patient_id'].map(patient_map)

fact_claim['patient_id_hashed'] = fact_claim['patient_id'].apply(hash_id)
patient_map = dict(zip(dim_patient['patient_id_hashed'], dim_patient['patient_sk']))
fact_claim['patient_sk'] = fact_claim['patient_id_hashed'].map(patient_map)


# Map provider_sk from dim_provider
provider_map = dict(zip(dim_provider['provider_id'], dim_provider['provider_sk']))
fact_claim['provider_sk'] = fact_claim['provider_id'].map(provider_map)

# Map proc_sk from dim_procedure
proc_map = dict(zip(dim_procedure['procedure_code'], dim_procedure['proc_sk']))
fact_claim['proc_sk'] = fact_claim['procedure_code'].map(proc_map)

# Drop original IDs (optional)
fact_claim = fact_claim[['claim_sk', 'patient_sk', 'provider_sk', 'proc_sk',
                         'date_of_service', 'billed_amount', 'paid_amount', 'adjudication_status']]

# Validate
print("✅ fact_claim table created successfully!")
print(fact_claim.head())

# Save to CSV
fact_claim.to_csv('fact_claim.csv', index=False)


✅ fact_claim table created successfully!
   claim_sk  patient_sk  provider_sk   proc_sk date_of_service  billed_amount  \
0         1           1          1.0  74476075      2023-10-13        5944.48   
1         2           2          2.0  69893948      2024-03-23        2014.75   
2         3           3          3.0  35867643      2023-11-23            NaN   
3         4           4          4.0  95044104      2024-12-05         502.79   
4         5           5          5.0  21477593      2024-08-23         418.11   

   paid_amount adjudication_status  
0      5880.39                Paid  
1      1926.35                Paid  
2      5168.72                Paid  
3          NaN      Partially Paid  
4          NaN                Paid  


In [21]:

# Add surrogate key first
fact_claim['claim_sk'] = range(1, len(fact_claim) + 1)

# Now run validation
validation_results = []

# 1. Uniqueness Checks
checks = {
    'dim_patient.patient_sk': dim_patient['patient_sk'].is_unique,
    'dim_provider.provider_sk': dim_provider['provider_sk'].is_unique,
    'dim_procedure.proc_sk': dim_procedure['proc_sk'].is_unique,
    'fact_claim.claim_sk': fact_claim['claim_sk'].is_unique
}
for name, result in checks.items():
    validation_results.append({'Check': f'Unique {name}', 'Status': 'PASS' if result else 'FAIL'})

# 2. Null Checks in Critical Columns
critical_fact_cols = ['claim_sk', 'patient_sk', 'provider_sk', 'proc_sk', 'date_of_service']
for col in critical_fact_cols:
    null_count = fact_claim[col].isnull().sum()
    validation_results.append({'Check': f'Nulls in fact_claim.{col}', 'Status': 'PASS' if null_count == 0 else f'FAIL ({null_count} nulls)'})

# 3. Referential Integrity Checks
invalid_patient_refs = fact_claim[~fact_claim['patient_sk'].isin(dim_patient['patient_sk'])]
validation_results.append({'Check': 'Referential integrity patient_sk', 'Status': 'PASS' if invalid_patient_refs.empty else f'FAIL ({len(invalid_patient_refs)} invalid)'})

invalid_provider_refs = fact_claim[~fact_claim['provider_sk'].isin(dim_provider['provider_sk'])]
validation_results.append({'Check': 'Referential integrity provider_sk', 'Status': 'PASS' if invalid_provider_refs.empty else f'FAIL ({len(invalid_provider_refs)} invalid)'})

invalid_proc_refs = fact_claim[~fact_claim['proc_sk'].isin(dim_procedure['proc_sk'])]
validation_results.append({'Check': 'Referential integrity proc_sk', 'Status': 'PASS' if invalid_proc_refs.empty else f'FAIL ({len(invalid_proc_refs)} invalid)'})

# Export validation report
report_df = pd.DataFrame(validation_results)
report_df.to_csv('validation_report.csv', index=False)

print("✅ Validation completed. Report saved as validation_report.csv")
print(report_df)


✅ Validation completed. Report saved as validation_report.csv
                                  Check              Status
0         Unique dim_patient.patient_sk                PASS
1       Unique dim_provider.provider_sk                PASS
2          Unique dim_procedure.proc_sk                PASS
3            Unique fact_claim.claim_sk                PASS
4          Nulls in fact_claim.claim_sk                PASS
5        Nulls in fact_claim.patient_sk                PASS
6       Nulls in fact_claim.provider_sk    FAIL (867 nulls)
7           Nulls in fact_claim.proc_sk                PASS
8   Nulls in fact_claim.date_of_service                PASS
9      Referential integrity patient_sk                PASS
10    Referential integrity provider_sk  FAIL (867 invalid)
11        Referential integrity proc_sk                PASS


In [35]:

import sqlite3
import pandas as pd

# Create an in-memory SQLite database
conn = sqlite3.connect(':memory:')

# Load your DataFrames into SQLite
dim_patient.to_sql('dim_patient', conn, index=False)
dim_provider.to_sql('dim_provider', conn, index=False)
dim_procedure.to_sql('dim_procedure', conn, index=False)
fact_claim.to_sql('fact_claim', conn, index=False)

# 1. Top Providers by Paid Amount

query1 = """
SELECT p.provider_id, p.speciality, SUM(f.paid_amount) AS total_paid
FROM fact_claim f
JOIN dim_provider p ON f.provider_sk = p.provider_sk
GROUP BY p.provider_id, p.speciality
ORDER BY total_paid DESC
LIMIT 10;
"""
result1 = pd.read_sql_query(query1, conn)
print("\nTop Providers by Paid Amount:")
print(result1)



# 2. Average Paid per Procedure
query2 = """
SELECT pr.procedure_code, pr.procedure_descrption, AVG(f.paid_amount) AS avg_paid
FROM fact_claim f
JOIN dim_procedure pr ON f.proc_sk = pr.proc_sk
GROUP BY pr.procedure_code, pr.procedure_descrption
ORDER BY avg_paid DESC;
"""
result2 = pd.read_sql_query(query2, conn)
print("\nAverage Paid per Procedure:")
print(result2)



Top Providers by Paid Amount:
    provider_id        speciality  total_paid
0  5.131668e+09  Vascular Surgery    39099.57
1  5.789238e+09           Urology    38956.62
2  7.136777e+09  Vascular Surgery    38594.60
3  9.948270e+09  Vascular Surgery    38411.20
4  3.994844e+09  Vascular Surgery    38263.84
5  3.370789e+09       Orthopedics    38236.55
6  4.906404e+09   General Surgery    38089.28
7  8.604238e+09            OB/GYN    38011.21
8  3.981756e+09  Vascular Surgery    37962.71
9  1.616153e+09       Orthopedics    37921.39

Average Paid per Procedure:
   procedure_code                        procedure_descrption      avg_paid
0           54520                                 Orchiectomy  19037.280208
1           35301                                Bypass graft  18927.046913
2           27447                            Knee replacement  18717.924576
3           49900                                Appendectomy  18667.271321
4           67028                             Retinal 

In [40]:

import pandas as pd
import numpy as np
import random

# Load the original fact_claim data
claims = pd.read_csv('fact_claim.csv')

# Prepare list to store line items
line_items = []

line_item_sk = 1

# Generate line items for each claim
for _, row in claims.iterrows():
    # Random number of line items per claim (1 to 3)
    num_items = random.randint(1, 4)
    for _ in range(num_items):
        units = random.randint(1, 5)
        # Handle NaN values in billed_amount and paid_amount
        billed_amount = row['billed_amount'] if pd.notnull(row['billed_amount']) else 0
        paid_amount = row['paid_amount'] if pd.notnull(row['paid_amount']) else 0
        # Distribute billed and paid amounts proportionally
        line_billed_amount = round(billed_amount * (random.uniform(0.2, 0.6)), 2)
        line_paid_amount = round(paid_amount * (random.uniform(0.2, 0.6)), 2)

        line_items.append([
            line_item_sk,
            row['claim_sk'],
            row['proc_sk'],
            units,
            line_billed_amount,
            line_paid_amount
        ])
        line_item_sk += 1

# Convert to DataFrame
line_item_df = pd.DataFrame(line_items, columns=[
    'line_item_sk', 'claim_sk', 'proc_sk', 'units', 'line_billed_amount', 'line_paid_amount'
])

# If we need exactly 75,000 records, sample or repeat
if len(line_item_df) < 75000:
    repeats = 75000 - len(line_item_df)
    extra_rows = line_item_df.sample(n=repeats, replace=True)
    line_item_df = pd.concat([line_item_df, extra_rows], ignore_index=True)
elif len(line_item_df) > 75000:
    line_item_df = line_item_df.sample(n=75000, random_state=42)

# Save to CSV
line_item_df.to_csv('fact_claim_lineitem_no_extra.csv', index=False)

print(f"Generated {len(line_item_df)} line items without extra columns and saved to fact_claim_lineitem_no_extra.csv")
print(line_item_df.head(10))


Generated 75000 line items without extra columns and saved to fact_claim_lineitem_no_extra.csv
   line_item_sk  claim_sk   proc_sk  units  line_billed_amount  \
0             1         1  74476075      4             2839.84   
1             2         1  74476075      3             3097.01   
2             3         1  74476075      3             3421.62   
3             4         2  69893948      2              434.68   
4             5         2  69893948      4              594.17   
5             6         2  69893948      5              427.03   
6             7         3  35867643      1                0.00   
7             8         3  35867643      4                0.00   
8             9         4  95044104      4              114.47   
9            10         4  95044104      3              121.10   

   line_paid_amount  
0           1203.91  
1           2318.46  
2           3194.93  
3            549.33  
4            466.58  
5            785.32  
6           2035.58  
7 